# Module 08: Building Custom Nipype Workflows

Nipype (Neuroimaging in Python — Pipelines and Interfaces) wraps command-line
neuroimaging tools in a common Python interface and chains them into
reproducible, parallelisable **directed acyclic graph (DAG)** workflows.

Instead of writing brittle shell scripts, you define **nodes** (one tool each)
and **connections** (data flow). Nipype tracks completed outputs, skips up-to-date
nodes on re-runs, and can distribute work across local cores or HPC schedulers.

## Learning objectives
1. Explain Nipype's node/workflow model.
2. Create `Node` and `Workflow` objects and configure interface inputs.
3. Connect nodes with `wf.connect()` to build a processing graph.
4. Build a minimal preprocessing workflow (BET → MCFLIRT → smooth).
5. Use `get_node_info()` to introspect workflow nodes.
6. Visualize workflow graphs with `workflow.write_graph()`.
7. Run workflows with the `Linear` and `MultiProc` plugins.

**Prerequisites:** Modules 00–07 completed; nipype ≥ 1.8; FSL ≥ 6.0 on `$PATH`.

**Estimated time:** ~3 hours.

## 1  What is Nipype?

Traditional neuroimaging pipelines are sequences of shell commands:

```bash
bet func.nii.gz func_brain -F -m
mcflirt -in func_brain.nii.gz -out func_mc -plots
fslmaths func_mc.nii.gz -s 2.548 func_smooth.nii.gz
```

This works but has problems: outputs are not tracked, re-running wastes time,
parallelisation is manual, and parameters are scattered across scripts.

Nipype solves this by representing each step as a **Node** and the whole
pipeline as a **Workflow** (a DAG). Key benefits:

| Feature | Description |
|---------|-------------|
| **Caching** | Completed nodes are skipped on re-runs (MD5 hash of inputs) |
| **Provenance** | Every command, output, and parameter is recorded |
| **Parallelism** | Independent nodes run simultaneously with `MultiProc` or HPC plugins |
| **Portability** | Same workflow object runs locally or on a cluster |
| **Introspection** | Inspect any node's interface, inputs, and outputs at any time |

In [ ]:
# Check that nipype is importable; print version information
try:
    import nipype
    import nipype.pipeline.engine as pe
    from nipype.interfaces import fsl
    from nipype.interfaces.utility import IdentityInterface
    print(f"nipype version : {nipype.__version__}")
except ImportError:
    print("nipype is not installed. Run: pip install nipype")
    print("You can still read this notebook, but code cells will not execute.")

# Check that FSL is on the PATH
import shutil, os
fsl_bin = shutil.which("bet")
if fsl_bin:
    print(f"FSL bet found  : {fsl_bin}")
    fsl_dir = os.environ.get("FSLDIR", "not set")
    print(f"FSLDIR         : {fsl_dir}")
else:
    print("FSL not found on PATH. Workflow nodes will fail at run time.")
    print("Install FSL: https://fsl.fmrib.ox.ac.uk/fsl/fslwiki/FslInstallation")

## 2  Core Concepts

### 2.1  Node

A `Node` wraps **one interface** (e.g., `fsl.BET`) and represents one
processing step. You configure inputs as Python attributes.

```python
bet = pe.Node(fsl.BET(frac=0.5, mask=True), name="bet")
bet.inputs.in_file = "/data/sub-01/anat/sub-01_T1w.nii.gz"
```

### 2.2  MapNode

A `MapNode` applies an interface to a **list** of inputs in parallel —
one job per element.

```python
bet_many = pe.MapNode(
    fsl.BET(mask=True),
    iterfield=["in_file"],
    name="bet_many",
)
bet_many.inputs.in_file = ["/sub-01/T1w.nii.gz", "/sub-02/T1w.nii.gz"]
```

### 2.3  IdentityInterface (inputnode / outputnode)

`IdentityInterface` is a pass-through node with named fields and no
computation. It gives workflows clean entry (`inputnode`) and exit
(`outputnode`) points.

### 2.4  Workflow and connect()

```python
wf = pe.Workflow(name="my_pipeline")
wf.base_dir = "/tmp/nipype_work"

wf.connect([
    (node_a, node_b, [("out_file", "in_file")]),
    (node_b, node_c, [("out_file", "in_file")]),
])
```

The third element is a list of `(source_field, dest_field)` tuples.
One `connect()` call can wire multiple nodes at once.

In [ ]:
# Demonstrate creating a simple Node and a two-node Workflow manually
try:
    import nipype.pipeline.engine as pe
    from nipype.interfaces import fsl
    from nipype.interfaces.utility import IdentityInterface

    # --- Create a standalone BET node ---
    bet_node = pe.Node(
        fsl.BET(frac=0.5, mask=True, functional=True),
        name="bet_standalone",
    )
    # Inspect available inputs without running
    print("BET node inputs (selected):")
    print(f"  in_file  : {bet_node.inputs.in_file}")
    print(f"  frac     : {bet_node.inputs.frac}")
    print(f"  mask     : {bet_node.inputs.mask}")

    # --- Build a tiny two-node workflow manually ---
    tiny_wf = pe.Workflow(name="tiny_example")

    inputnode = pe.Node(
        IdentityInterface(fields=["func"]),
        name="inputnode",
    )
    mcflirt_node = pe.Node(
        fsl.MCFLIRT(mean_vol=True, save_plots=True),
        name="mcflirt",
    )

    tiny_wf.connect([
        (inputnode, mcflirt_node, [("func", "in_file")]),
    ])

    print("\nTiny workflow nodes:",
          [n.name for n in tiny_wf._graph.nodes()])

except ImportError:
    print("nipype not available — skipping demo.")

## 3  Creating the Minimal Preprocessing Workflow

`utils/nipype_helpers.py` provides `create_minimal_preproc_workflow()`, which
builds an FSL pipeline with four nodes:

```
inputnode
  │  func ──► meanvol (fsl.MeanImage)
  │              │  out_file ──► bet (fsl.BET)  →  brain_mask
  │  func ──► mcflirt (fsl.MCFLIRT)  →  motion_params
  │              │  out_file ──► smooth (fsl.IsotropicSmooth)
  │  fwhm ──────────────────────────────────────►  smooth
  ▼
outputnode  (preprocessed_func, motion_params, brain_mask)
```

The function returns a configured (but not yet run) `Workflow` object.

In [ ]:
import os
import sys

# Add repo root to sys.path so we can import from utils/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

try:
    from utils.nipype_helpers import (
        create_minimal_preproc_workflow,
        create_first_level_workflow,
        get_node_info,
        run_workflow,
    )
    print("Successfully imported nipype_helpers utilities.")
except ImportError as e:
    print(f"Import error: {e}")
    print("Ensure you are running from within the module_08_nipype_workflows/ directory")
    print("and that nipype is installed.")

In [ ]:
# Create the minimal preprocessing workflow
try:
    preproc_wf = create_minimal_preproc_workflow(name="minimal_preproc")
    print(f"Workflow created : {preproc_wf.name}")
    print(f"Node count       : {len(list(preproc_wf._graph.nodes()))}")
    print(f"Nodes            : {[n.name for n in preproc_wf._graph.nodes()]}")
except Exception as e:
    print(f"Could not create workflow: {e}")

## 4  Inspecting the Workflow with `get_node_info()`

`get_node_info(workflow)` iterates over every node in the workflow's graph
and prints its name, wrapped interface class, and available input fields.
It returns a list of dicts so you can query it programmatically.

Use this to verify connections are correct before running — it is far
cheaper than a failed workflow run.

In [ ]:
try:
    node_info = get_node_info(preproc_wf)

    # Programmatic access: find the mcflirt node's inputs
    mcflirt_info = next(
        (n for n in node_info if n["name"] == "mcflirt"), None
    )
    if mcflirt_info:
        print("\nMCFLIRT inputs (first 10):")
        for inp in mcflirt_info["inputs"][:10]:
            print(f"  {inp}")
except NameError:
    print("preproc_wf not defined — run Section 3 first.")

## 5  Visualizing the Workflow Graph

`workflow.write_graph()` uses graphviz to render the DAG as a PNG.
Two graph styles are available:

| `graph2use` | Description |
|-------------|-------------|
| `'orig'` | One node per Nipype Node |
| `'flat'` | Expand nested workflows inline |
| `'exec'` | Show only execution-relevant nodes |
| `'hierarchical'` | Nested subgraph layout |

The output files (`graph.dot`, `graph.dot.png`) appear in the workflow's
working directory.

In [ ]:
import os
from IPython.display import Image, display

GRAPH_DIR = "/tmp/nipype_graphs"
os.makedirs(GRAPH_DIR, exist_ok=True)

try:
    preproc_wf.base_dir = GRAPH_DIR

    graph_path = preproc_wf.write_graph(
        graph2use="orig",
        format="png",
        simple_form=True,
    )
    print(f"Graph written to: {graph_path}")

    # Display inline in the notebook
    if os.path.exists(graph_path):
        display(Image(graph_path))
    else:
        # write_graph may return the .dot path; look for the .png
        png_path = graph_path.replace(".dot", ".dot.png")
        if os.path.exists(png_path):
            display(Image(png_path))

except Exception as e:
    # graphviz is optional — workflow still runs without it
    print(f"Graph visualization skipped: {e}")
    print("Install graphviz to enable: conda install graphviz")

## 6  Setting Inputs and Running with the Linear Plugin

Before running, you must:
1. Set `workflow.base_dir` — all intermediate files go here.
2. Set `inputnode.inputs.*` for every required field.

The `Linear` plugin executes nodes one at a time in topological order.
It is the safest choice for debugging because errors are easy to read.

> **Note:** Replace `BOLD_FILE` with a real 4-D NIfTI path on your system.
> The cell below uses a synthetic path; it will error if FSL is installed
> but the file does not exist.

In [ ]:
import os

# Replace with the path to a real preprocessed BOLD NIfTI on your system
BOLD_FILE = "/data/bids/sub-01/func/sub-01_task-rest_bold.nii.gz"
OUTPUT_DIR = "/tmp/nipype_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    # Re-create a fresh workflow so we start clean
    wf_linear = create_minimal_preproc_workflow(name="preproc_linear")
    wf_linear.base_dir = OUTPUT_DIR

    # Set required inputs on the inputnode
    inputnode = wf_linear.get_node("inputnode")
    inputnode.inputs.func = BOLD_FILE
    inputnode.inputs.fwhm = 6.0  # smoothing kernel in mm

    print(f"Workflow   : {wf_linear.name}")
    print(f"base_dir   : {wf_linear.base_dir}")
    print(f"func input : {inputnode.inputs.func}")
    print(f"fwhm input : {inputnode.inputs.fwhm} mm")

    if os.path.exists(BOLD_FILE):
        # Run with the Linear (serial) plugin
        run_workflow(wf_linear, plugin="Linear")
    else:
        print(f"\nFile not found: {BOLD_FILE}")
        print("Update BOLD_FILE to point to a real NIfTI to execute.")

except NameError:
    print("create_minimal_preproc_workflow not imported — run Section 3 first.")
except Exception as e:
    print(f"Workflow setup error: {e}")

## 7  Running with the MultiProc Plugin

For datasets with many independent nodes (e.g., when iterating over subjects
or runs), `MultiProc` distributes work across CPU cores using Python's
`multiprocessing` module.

```python
wf.run(plugin="MultiProc", plugin_args={"n_procs": 8})
```

Guidelines:
- Set `n_procs` ≤ the number of physical cores (use `os.cpu_count()`).
- Each FSL tool can itself be multi-threaded; avoid over-subscribing CPUs.
- For a single subject with a small workflow, `MultiProc` overhead may
  exceed the benefit — profile before committing.
- On HPC systems, prefer the `SLURM` or `SGE` plugin instead.

In [ ]:
import os

# Determine a safe number of processes for this machine
n_cpus = os.cpu_count() or 1
n_procs = max(1, n_cpus // 2)  # use half the cores to leave headroom
print(f"Available CPUs : {n_cpus}")
print(f"Using n_procs  : {n_procs}")

try:
    wf_multi = create_minimal_preproc_workflow(name="preproc_multiproc")
    wf_multi.base_dir = OUTPUT_DIR

    inputnode_m = wf_multi.get_node("inputnode")
    inputnode_m.inputs.func = BOLD_FILE
    inputnode_m.inputs.fwhm = 6.0

    if os.path.exists(BOLD_FILE):
        run_workflow(wf_multi, plugin="MultiProc", n_procs=n_procs)
    else:
        print(f"File not found: {BOLD_FILE}")
        print("Update BOLD_FILE above to a real NIfTI to execute.")

        # Show what the call would look like
        print("\nEquivalent direct call:")
        print(f"  wf_multi.run(plugin='MultiProc',")
        print(f"               plugin_args={{'n_procs': {n_procs}}})")

except NameError:
    print("create_minimal_preproc_workflow not imported — run Section 3 first.")
except Exception as e:
    print(f"Workflow error: {e}")

## 8  Creating the First-Level GLM Workflow

`create_first_level_workflow()` builds an FSL FEAT-based GLM pipeline:

```
inputnode (func, events, confounds, TR)
  │
  ├──► specify_model  (nipype SpecifyModel)
  │         │  session_info
  │         ▼
  ├──► level1design  (fsl.Level1Design)
  │         │  fsf_files, ev_files
  │         ▼
  ├──► feat_model    (fsl.FEATModel)
  │         │  design_file, con_file
  │         ▼
  └──► filmgls       (fsl.FILMGLS)
            │
            ▼
       outputnode (stats_dir, dof_file)
```

This workflow produces a `stats/` directory containing parameter estimate
images (`pe*.nii.gz`), t-statistic maps (`tstat*.nii.gz`), and a
degrees-of-freedom file (`dof`).

In [ ]:
try:
    glm_wf = create_first_level_workflow(name="first_level_glm")
    print(f"GLM workflow created : {glm_wf.name}")
    print(f"Nodes                : {[n.name for n in glm_wf._graph.nodes()]}")

    # Inspect GLM workflow nodes
    glm_info = get_node_info(glm_wf)

except NameError:
    print("Helper functions not imported — run Section 3 first.")
except Exception as e:
    print(f"Error creating GLM workflow: {e}")

In [ ]:
# Example: configure the GLM workflow with a real preprocessed BOLD run
# Replace these paths with real files from your fMRIPrep output directory

PREPROC_BOLD = (
    "/data/fmriprep/sub-01/func/"
    "sub-01_task-rest_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz"
)
EVENTS_TSV  = "/data/bids/sub-01/func/sub-01_task-rest_events.tsv"
CONFOUNDS   = (
    "/data/fmriprep/sub-01/func/"
    "sub-01_task-rest_desc-confounds_timeseries.tsv"
)
TR = 2.0  # repetition time in seconds

try:
    glm_wf.base_dir = OUTPUT_DIR

    glm_inputnode = glm_wf.get_node("inputnode")
    glm_inputnode.inputs.func      = PREPROC_BOLD
    glm_inputnode.inputs.events    = EVENTS_TSV
    glm_inputnode.inputs.confounds = CONFOUNDS
    glm_inputnode.inputs.TR        = TR

    print("GLM workflow inputs configured:")
    print(f"  func      : {glm_inputnode.inputs.func}")
    print(f"  events    : {glm_inputnode.inputs.events}")
    print(f"  confounds : {glm_inputnode.inputs.confounds}")
    print(f"  TR        : {glm_inputnode.inputs.TR} s")

    if os.path.exists(PREPROC_BOLD) and os.path.exists(EVENTS_TSV):
        run_workflow(glm_wf, plugin="Linear")
    else:
        print("\nInput files not found — update paths above to run.")

except NameError:
    print("glm_wf not defined — run the cell above first.")
except Exception as e:
    print(f"GLM workflow error: {e}")

## 9  Practical Tips

### Working directories

Set `workflow.base_dir` to a directory with plenty of storage — nipype
writes every intermediate file. A single subject preprocessing run can
produce several GB of intermediate data.

```
<base_dir>/
└── <workflow_name>/
    ├── inputnode/
    ├── meanvol/
    │   └── _report/
    ├── bet/
    ├── mcflirt/
    └── smooth/
```

### Crash files

If a node fails, nipype writes a crash file to `<base_dir>/crash/`:

```bash
# Inspect a crash file
nipypecli crash crash-<hash>.pklz
```

Common causes: FSL not on `$PATH`, missing input file, wrong field name.

### Re-running after changes

Nipype uses MD5 hashes of input parameters to decide whether to skip a node.
If you change an input (e.g., `fwhm`), only the downstream nodes re-run.
To force a full re-run, delete the working directory or call:

```python
wf.config['execution']['stop_on_first_rerun'] = False
```

### Debugging a single node

Run a node in isolation before wiring it into a workflow:

```python
from nipype.interfaces import fsl
bet = fsl.BET(in_file="/data/func.nii.gz", frac=0.3, mask=True)
result = bet.run()
print(result.outputs)
```

### Choosing a plugin

| Context | Recommended plugin |
|---------|-------------------|
| Debugging / single subject | `Linear` |
| Multi-subject on workstation | `MultiProc` |
| HPC SLURM cluster | `SLURM` |
| HPC SGE cluster | `SGE` |

## 10  Summary

In this module you have:

- Learned how Nipype represents pipelines as directed acyclic graphs of **nodes**
  connected by explicit data-flow edges.
- Created `Node` and `Workflow` objects and configured interface inputs.
- Built a minimal preprocessing workflow (mean volume → BET → MCFLIRT → smooth)
  using `create_minimal_preproc_workflow()`.
- Used `get_node_info()` to introspect every node's interface and inputs.
- Visualized the workflow DAG with `write_graph()`.
- Run the workflow with both the `Linear` (serial) and `MultiProc` (parallel)
  execution plugins.
- Seen how the same pattern extends to a first-level GLM workflow with
  `create_first_level_workflow()`.

### Next steps

- Explore the `scripts/` directory for ready-to-use command-line tools.
- Adapt `create_minimal_preproc_workflow()` to add slice-timing correction
  (`fsl.SliceTimer`) or ICA-AROMA.
- Read the Nipype documentation for `MapNode`, `iterables`, and the
  `DataSink` node (for organizing outputs).
- For production analyses, consider [fMRIPrep](https://fmriprep.org)
  (which is itself built on Nipype) for fully validated preprocessing.

### References

- Gorgolewski K, et al. (2011). Nipype: a flexible, lightweight and extensible
  neuroimaging data processing framework in Python. *Frontiers in Neuroinformatics*,
  5, 13. https://doi.org/10.3389/fninf.2011.00013
- Nipype documentation: https://nipype.readthedocs.io
- FSL documentation: https://fsl.fmrib.ox.ac.uk/fsl/fslwiki